# Trafilatura: Overview

A hands-on tour of the main features. For full documentation see [trafilatura.readthedocs.io](https://trafilatura.readthedocs.io/).

## 1. Installation

```bash
pip install trafilatura
```

To update: `pip install -U trafilatura`. For more info see [Installation](https://trafilatura.readthedocs.io/en/latest/installation.html).

## 2. Quick start

Download a web page and extract its main text:

In [ ]:
from trafilatura import extract, fetch_url

url = "https://github.blog/2019-03-29-leader-spotlight-erin-spiceland/"
downloaded = fetch_url(url)

text = extract(downloaded)
print(text[:500])

## 3. Input formats

`extract()` accepts HTML as a string or as a pre-parsed LXML tree.

In [ ]:
# from a local file
# with open('myfile.html', encoding='utf-8') as f:
#     text = extract(f.read())

# from an HTML string
my_html = "<html><body><article><p>This is the main text.</p></article></body></html>"
print(extract(my_html))

In [ ]:
# from a pre-parsed LXML tree
from lxml import html

tree = html.fromstring(downloaded)
print(extract(tree)[:200])

## 4. Output formats

Available formats: `txt` (default), `json`, `markdown`, `xml`, `xmltei`, `csv`, `html`.

In [ ]:
# JSON with metadata (metadata is off by default)
result = extract(downloaded, output_format="json", with_metadata=True)
print(result[:500])

In [ ]:
# Markdown
result = extract(downloaded, output_format="markdown")
print(result[:500])

In [ ]:
# XML
result = extract(downloaded, output_format="xml")
print(result[:500])

## 5. Extraction options

### Extraction modes

Trafilatura offers several extraction strategies. Compare them on the same page:

In [ ]:
standard = extract(downloaded)
fast     = extract(downloaded, fast=True)
precise  = extract(downloaded, favor_precision=True)
recall   = extract(downloaded, favor_recall=True)

for label, result in [("standard", standard), ("fast", fast), ("precision", precise), ("recall", recall)]:
    print(f"{label:>10}: {len(result):>5} chars")

### Including or excluding elements

Comments and tables are included by default. Formatting and images are not.

In [ ]:
result = extract(downloaded,
    include_comments=False,
    include_tables=False,
    include_formatting=True,
    include_images=True,
    output_format="markdown",
)
print(result[:500])

### Reusable settings with `Extractor`

Bundle options into an `Extractor` object to reuse across multiple extractions:

In [ ]:
from trafilatura.settings import Extractor

options = Extractor(
    output_format="json",
    with_metadata=True,
    comments=False,
)

# reuse on multiple pages
result = extract(downloaded, options=options)
print(result[:300])

## 6. Other extraction functions

| Function | Returns | Use case |
|----------|---------|----------|
| `extract()` | string | Main extraction, all formats |
| `bare_extraction()` | `Document` object | Structured access to fields |
| `html2txt()` | string | All text (including nav, footers) |
| `extract_metadata()` | `Document` object | Metadata only |

In [ ]:
from trafilatura import bare_extraction

doc = bare_extraction(downloaded, with_metadata=True)
print(f"Title:  {doc.title}")
print(f"Author: {doc.author}")
print(f"Date:   {doc.date}")
print(f"Text:   {doc.text[:100]}...")

In [ ]:
from trafilatura import html2txt

# extract ALL text, including navigation, footers, etc.
all_text = html2txt(downloaded)
print(f"html2txt: {len(all_text):>5} chars (everything)")
print(f"extract:  {len(extract(downloaded)):>5} chars (main content only)")

In [ ]:
from trafilatura import extract_metadata

meta = extract_metadata(downloaded)
print(f"Title:    {meta.title}")
print(f"Author:   {meta.author}")
print(f"Date:     {meta.date}")
print(f"Sitename: {meta.sitename}")

## 7. URL discovery

Three strategies to find URLs on a website:

- **Feeds** (Atom/RSS): fresh content
- **Sitemaps**: exhaustive listing by the site owner
- **Crawling**: follow internal links

In [ ]:
from trafilatura.feeds import find_feed_urls

links = find_feed_urls("https://www.theguardian.com/")
print(f"{len(links)} URLs found")
print("\n".join(links[:5]))

In [ ]:
from trafilatura.sitemaps import sitemap_search

links = sitemap_search("https://www.sitemaps.org")
print(f"{len(links)} URLs found")
print("\n".join(links[:5]))

In [ ]:
from trafilatura.spider import focused_crawler

to_visit, known_urls = focused_crawler("https://www.python.org", max_seen_urls=10)
print(f"{len(to_visit)} URLs to visit, {len(known_urls)} URLs seen")

## 8. Language filtering

Filter results by language using ISO 639-1 codes. If the detected language doesn't match, `extract()` returns `None`.

> Requires: `pip install trafilatura[all]`

In [ ]:
# English page, filtering for English → should return text
result_en = extract(downloaded, target_language="en")
print(f"English filter: {len(result_en) if result_en else 0} chars")

# same page, filtering for Japanese → should return None
result_ja = extract(downloaded, target_language="ja")
print(f"Japanese filter: {result_ja}")

## 9. Deduplication

Detect near-duplicate content using Simhash fingerprints. Useful when building a corpus from a single site where boilerplate repeats across pages.

In [ ]:
from trafilatura.deduplication import Simhash, content_fingerprint

text_a = extract(downloaded)
text_b = text_a  # identical
text_c = "Completely different content about something else entirely."

hash_a = Simhash(text_a)
hash_b = Simhash(text_b)
hash_c = Simhash(text_c)

print(f"a vs b (identical):  similarity={hash_a.similarity(hash_b):.3f}")
print(f"a vs c (different):  similarity={hash_a.similarity(hash_c):.3f}")
print(f"\nFingerprint: {content_fingerprint(text_a)}")

## 10. Putting it together

A mini pipeline: discover URLs → download → extract text.

In [ ]:
from trafilatura import extract, fetch_url
from trafilatura.feeds import find_feed_urls

# discover URLs from a feed
urls = find_feed_urls("https://www.theguardian.com/")
print(f"Found {len(urls)} URLs")

# download and extract the first 3
for url in urls[:3]:
    html = fetch_url(url)
    if html:
        text = extract(html, favor_precision=True)
        if text:
            print(f"\n--- {url} ---")
            print(f"{len(text)} chars extracted")
            print(text[:200] + "...")

---

Trafilatura is also available as a **command-line tool**: `trafilatura -u "https://example.org" --json`

See the [CLI documentation](https://trafilatura.readthedocs.io/en/latest/usage-cli.html) and the full [Python usage guide](https://trafilatura.readthedocs.io/en/latest/usage-python.html) for more.